<h3 style="color: white">Inconsistent Data Entry</h3>

<h4 style="color: red">Get our environment set up</h4>

In [2]:
# modules we'll use
import pandas as pd
import numpy as np

# helpful modules
import rapidfuzz
import charset_normalizer

# read in all our data
professors = pd.read_csv("pakistan_intellectual_capital.csv")

<h4 style="color: red">Do some preliminary text pre-processing</h4>

<p style="font-size: 20px">We'll begin by taking a quick look at the first few rows of the data.</p>

In [3]:
professors.head()

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN


<p style="font-size: 20px"><b>Say we're interested in cleaning up the "Country" column to make sure there's no data entry inconsistencies</b> in it. We could go through and check each row by hand, of course, and hand-correct inconsistencies when we find them. There's a more efficient way to do this, though!</p>

In [4]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries=sorted(countries)
print(countries[0:30])

[' Germany', ' New Zealand', ' Sweden', ' USA', 'Australia', 'Austria', 'Canada', 'China', 'Finland', 'France', 'Greece', 'HongKong', 'Ireland', 'Italy', 'Japan', 'Macau', 'Malaysia', 'Mauritius', 'Netherland', 'New Zealand', 'Norway', 'Pakistan', 'Portugal', 'Russian Federation', 'Saudi Arabia', 'Scotland', 'Singapore', 'South Korea', 'SouthKorea', 'Spain']


<p style="font-size: 20px">Just looking at this, I can see some problems due to inconsistent data entry: ' Germany', or ' New Zealand' and 'New Zealand'.v</p>
<p style="font-size: 20px"><b style="color: green">The first thing I'm going to do is make everything lower case (I can change it back at the end) and remove any white spaces at the beginning and end of cells.</b><br><br>
Inconsistencies in capitalizations and trailing white spaces are very common in text data and you can fix a good 80% of your text data entry inconsistencies by doing this.</p>

In [5]:
# convert to lower case
professors['Country'] = professors['Country'].str.lower()
# remove trailing white spaces
professors['Country'] = professors['Country'].str.strip()

<h4 style="color: red">Use fuzzy matching to correct inconsistent data entry</h4>

<p style="font-size: 20px">Alright, let's take another look at the 'Country' column and see if there's any more data cleaning we need to do.</p>

In [6]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries=sorted(countries)
print(countries[0:30])

['australia', 'austria', 'canada', 'china', 'finland', 'france', 'germany', 'greece', 'hongkong', 'ireland', 'italy', 'japan', 'macau', 'malaysia', 'mauritius', 'netherland', 'new zealand', 'norway', 'pakistan', 'portugal', 'russian federation', 'saudi arabia', 'scotland', 'singapore', 'south korea', 'southkorea', 'spain', 'sweden', 'thailand', 'turkey']


<p style="font-size: 20px">It does look like there is another inconsistency: <span style="color: red">'southkorea'</span> and <span style="color: red">'south korea'</span> should be the same.</p>

<p style="font-size: 20px">We're going to use the <b style="color: green">rapidfuzz</b> package to help identify which strings are closest to each other.</p>

<details>
<summary><b style="color: green">Fuzzy matching</b></summary>
<p style="font-size: 20px">The process of automatically finding text strings that are very similar to the target string.</p>
<p style="font-size: 20px">In general, a string is considered "closer" to another one if the fewer characters are changed if you were transforming one string into another. So <span style="color: red">"apple" and "snapple"</span> are two changes away from each other (add "s" and "n") while <span style="color: red">"in" and "on"</span> and one change away (rplace "i" with "o").</p>
<p style="font-size: 20px">You won't always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.</p>
</details>

<p style="font-size: 20px"><b style="color: green"><code>rapidfuzz</code> returns a ratio given two strings. The closer the ratio is to 100, the smaller the edit distance between the two strings.</b><br><br>
Here, we're going to get the ten strings from our list of cities that have the closest distance to "south korea".</p>

In [7]:
# get the top 10 closest matches to "south korea"
matches = rapidfuzz.process.extract('south korea', countries, limit=10, scorer=rapidfuzz.fuzz.token_sort_ratio)

# take a look at them
matches

[('south korea', 100.0, 24),
 ('southkorea', 47.61904761904761, 25),
 ('saudi arabia', 43.47826086956522, 21),
 ('norway', 35.29411764705882, 17),
 ('austria', 33.333333333333336, 1),
 ('ireland', 33.333333333333336, 9),
 ('pakistan', 31.57894736842105, 18),
 ('portugal', 31.57894736842105, 19),
 ('scotland', 31.57894736842105, 22),
 ('australia', 30.000000000000004, 0)]

<p style="font-size: 20px">We can see that two of the items in the cities are very close to <span style="color: red">"south korea": "south korea" and "southkorea".</span><br><br>
Let's replace all rows in our "Country" column that <span style="color: red">have a ratio of > 47 with "south korea".</span></p>

<p style="font-size: 20px">To do this, I'm going to write a function. (It's a good idea to write a general purpose function you can reuse if you think you might have to do a specific task more than once or twice.</p>

In [ ]:
def replace_matches_in_column(df: pd.DataFrame, column, string_to_match, min_ratio=47):
    #get the list of unique strings:
    strings=df[column].unique()

    # get the top 10 closest matches to string_to_match
    matches = rapidfuzz.process.extract(string_to_match, countries, limit=10, scorer=rapidfuzz.fuzz.token_sort_ratio)

    closest_matches=[t[0] for t in matches if t[1]>min_ratio] # only get matches with a ratio > min_ratio

    row_with_closes_matches=df[column].isin(closest_matches) # get the rows of all the close matches in our dataframe

    df.loc[row_with_closes_matches, column]=string_to_match # replace all rows with close matches with the input matches 

    print('Matching done.')

<p style="font-size: 20px">Now that we have a function, we can put it to the test!</p>

In [9]:
# use the function we just wrote to replace close matches to "south korea" with "south korea"
replace_matches_in_column(df=professors, column='Country', string_to_match="south korea")

Matching done.


<p style="font-size: 20px">And now let's check the unique values in our "Country" column again and make sure we've tidied up "south korea" correctly.</p>

In [10]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries=sorted(countries)
print(countries[0:30])

['australia', 'austria', 'canada', 'china', 'finland', 'france', 'germany', 'greece', 'hongkong', 'ireland', 'italy', 'japan', 'macau', 'malaysia', 'mauritius', 'netherland', 'new zealand', 'norway', 'pakistan', 'portugal', 'russian federation', 'saudi arabia', 'scotland', 'singapore', 'south korea', 'spain', 'sweden', 'thailand', 'turkey', 'uk']


<p style="font-size: 20px">Excellent! Now we only have "south korea" in our dataframe and we didn't have to change anything by hand.</p>